In [1]:
import json
import os
import time
from typing import Literal, TypedDict

import arxiv
import psycopg2
import requests
import wikipedia
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from mem0 import MemoryClient
from pgvector.psycopg2 import register_vector
from psycopg2.extras import Json
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

load_dotenv()

for key in [
    "GROQ_API_KEY",
    "OPENALEX_API_KEY",
    "SUPABASE_DB_URL",
    "MEM0_API_KEY",
    "TAVILY_API_KEY",
]:
    assert os.environ.get(key), f"Missing {key} in .env"

print("Environment loaded ✅")

Environment loaded ✅


In [2]:
class Paper(BaseModel):
    title: str
    authors: list[str]
    year: int | None = None
    abstract: str | None = None
    url: str | None = None
    pdf_url: str | None = None
    citation_count: int | None = None
    source: Literal["arxiv", "openalex", "semantic_scholar", "wikipedia"]

### Tools

In [3]:
def search_arxiv(query: str, max_results: int = 5) -> list[Paper]:
    client = arxiv.Client(page_size=max_results, delay_seconds=3.0, num_retries=2)
    search = arxiv.Search(
        query=query, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance
    )
    try:
        return [
            Paper(
                title=r.title,
                authors=[a.name for a in r.authors],
                year=r.published.year,
                abstract=r.summary.replace("\n", " "),
                url=r.entry_id,
                pdf_url=r.pdf_url,
                source="arxiv",
            )
            for r in client.results(search)
        ]
    except Exception as e:  # noqa: BLE001
        print(f"⚠️ arXiv search failed ({type(e).__name__}); continuing without it")
        return []


OPENALEX_BASE = "https://api.openalex.org/works"


def _reconstruct_abstract(inverted_index: dict | None) -> str | None:
    if not inverted_index:
        return None
    positions = {}
    for word, idxs in inverted_index.items():
        for idx in idxs:
            positions[idx] = word
    return " ".join(positions[i] for i in sorted(positions))


def search_openalex(query: str, max_results: int = 5) -> list[Paper]:
    params = {
        "search": query,
        "per_page": max_results,
        "select": "title,authorships,publication_year,abstract_inverted_index,id,open_access,cited_by_count",
        "api_key": os.environ["OPENALEX_API_KEY"],
    }
    try:
        resp = requests.get(OPENALEX_BASE, params=params, timeout=15)
        resp.raise_for_status()
    except Exception as e:  # noqa: BLE001
        print(f"⚠️ OpenAlex search failed ({type(e).__name__}); continuing without it")
        return []

    results = []
    for w in resp.json()["results"]:
        results.append(
            Paper(
                title=w.get("title") or "Untitled",
                authors=[a["author"]["display_name"] for a in w.get("authorships", [])],
                year=w.get("publication_year"),
                abstract=_reconstruct_abstract(w.get("abstract_inverted_index")),
                url=w.get("id"),
                pdf_url=(w.get("open_access") or {}).get("oa_url"),
                citation_count=w.get("cited_by_count"),
                source="openalex",
            )
        )
    return results


_last_ss_call = 0.0
SEMANTIC_SCHOLAR_BASE = "https://api.semanticscholar.org/graph/v1/paper/search"


def search_semantic_scholar(query: str, max_results: int = 5) -> list[Paper]:
    global _last_ss_call
    elapsed = time.time() - _last_ss_call
    if elapsed < 1.0:
        time.sleep(1.0 - elapsed)
    _last_ss_call = time.time()

    headers = {"x-api-key": os.environ.get("SEMANTIC_SCHOLAR_API_KEY", "")}
    params = {
        "query": query,
        "limit": max_results,
        "fields": "title,year,abstract,authors,citationCount,openAccessPdf",
    }
    try:
        resp = requests.get(
            SEMANTIC_SCHOLAR_BASE, params=params, headers=headers, timeout=15
        )
        resp.raise_for_status()
        data = resp.json().get("data", [])
    except Exception as e:  # noqa: BLE001
        print(f"⚠️ Semantic Scholar search failed ({type(e).__name__}); skipping")
        return []

    return [
        Paper(
            title=p.get("title") or "Untitled",
            authors=[a["name"] for a in p.get("authors", [])],
            year=p.get("year"),
            abstract=p.get("abstract"),
            url=f"https://www.semanticscholar.org/paper/{p['paperId']}"
            if p.get("paperId")
            else None,
            pdf_url=(p.get("openAccessPdf") or {}).get("url"),
            citation_count=p.get("citationCount"),
            source="semantic_scholar",
        )
        for p in data
    ]


# Remove: import wikipedia  (can `uv remove wikipedia`, no longer needed)

WIKIPEDIA_API_BASE = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_HEADERS = {
    "User-Agent": "LearnLoop/1.0 (personal learning project; contact: your-email@example.com)"
}


def search_wikipedia(query: str, max_results: int = 2) -> list[Paper]:
    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": query,
        "gsrlimit": max_results,
        "prop": "extracts|info",
        "exintro": True,
        "explaintext": True,
        "inprop": "url",
        "format": "json",
    }
    try:
        resp = requests.get(
            WIKIPEDIA_API_BASE, params=params, headers=WIKIPEDIA_HEADERS, timeout=15
        )
        resp.raise_for_status()
        pages = resp.json().get("query", {}).get("pages", {})
    except Exception as e:
        print(f"⚠️ Wikipedia search failed ({type(e).__name__}); skipping")
        return []

    return [
        Paper(
            title=page.get("title", "Untitled"),
            authors=["Wikipedia contributors"],
            year=None,
            abstract=page.get("extract"),
            url=page.get("fullurl"),
            pdf_url=None,
            citation_count=None,
            source="wikipedia",
        )
        for page in pages.values()
        if page.get("extract")
    ]

### RAG

In [4]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")


def _connect():
    conn = psycopg2.connect(
        os.environ["SUPABASE_DB_URL"],
        connect_timeout=10,
        options="-c statement_timeout=15000",
    )
    conn.autocommit = True
    register_vector(conn)
    return conn


conn = _connect()
print("Connected to Supabase ✅")


def ensure_connection():
    global conn
    try:
        with conn.cursor() as cur:
            cur.execute("select 1")
    except Exception:
        conn = _connect()


splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)


def chunk_paper(paper: Paper) -> list[str]:
    text = paper.abstract or ""
    return splitter.split_text(text) if text else []


def ingest_papers(papers: list[Paper]) -> int:
    ensure_connection()
    inserted = 0
    with conn.cursor() as cur:
        for paper in papers:
            chunks = chunk_paper(paper)
            if not chunks:
                continue
            embeddings = embedder.encode(chunks)
            for chunk_text, embedding in zip(chunks, embeddings):
                cur.execute(
                    """insert into paper_chunks
                       (paper_title, paper_url, source, chunk_text, embedding, metadata)
                       values (%s, %s, %s, %s, %s::vector, %s)""",
                    (
                        paper.title,
                        paper.url,
                        paper.source,
                        chunk_text,
                        embedding,
                        Json(
                            {"year": paper.year, "citation_count": paper.citation_count}
                        ),
                    ),
                )
                inserted += 1
    return inserted


def retrieve_context(query: str, top_k: int = 5) -> list[dict]:
    ensure_connection()
    query_embedding = embedder.encode([query])[0]
    with conn.cursor() as cur:
        cur.execute(
            "select * from match_paper_chunks(%s::vector, %s)", (query_embedding, top_k)
        )
        cols = [d[0] for d in cur.description]
        return [dict(zip(cols, row)) for row in cur.fetchall()]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connected to Supabase ✅


### LLM & Mem0

In [5]:
USER_ID = "prashant"

llm = ChatGroq(
    model="openai/gpt-oss-120b", temperature=0.3, api_key=os.environ["GROQ_API_KEY"]
)
mem0_client = MemoryClient(api_key=os.environ["MEM0_API_KEY"])


def get_learner_context(topic: str, limit: int = 10) -> str:
    response = mem0_client.search(
        f"knowledge gaps, quiz mistakes, and learning preferences for {topic}",
        filters={"user_id": USER_ID},
        limit=limit,
    )
    memories = response.get("results", []) if isinstance(response, dict) else response
    if not memories:
        return "No prior context on this topic yet — first time it's coming up."
    return "\n".join(f"- {m['memory']}" for m in memories)


def store_correction(topic: str, user_reaction: str):
    messages = [
        {"role": "user", "content": f"Regarding the topic '{topic}': {user_reaction}"}
    ]
    mem0_client.add(messages, user_id=USER_ID, metadata={"topic": topic})

### Explainer & Critique

In [6]:
EXPLAINER_SYSTEM_PROMPT = """You are a personal tutor explaining a concept to one specific learner.
Use ONLY the retrieved source material to ground your explanation — never invent facts.
The 'What I know about this learner' section may list specific gaps or mistakes from past quizzes.
If it does, your explanation MUST directly and explicitly address those specific gaps — do not give a
generic overview instead. If there's no prior context, default to a clear, moderately technical
explanation with a concrete example."""


def explain_topic(topic: str) -> dict:
    sources = retrieve_context(topic, top_k=5)
    source_text = "\n\n".join(
        f"[{s['paper_title']}]: {s['chunk_text']}" for s in sources
    )
    learner_context = get_learner_context(topic)

    prompt = f"""Topic: {topic}

Retrieved source material:
{source_text}

What I know about this learner:
{learner_context}

Explain this topic to the learner now."""

    response = llm.invoke(
        [
            {"role": "system", "content": EXPLAINER_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
    )
    return {"explanation": response.content, "sources": sources}


CRITIQUE_SYSTEM_PROMPT = """You are a fact-checker. Compare the explanation against the source material.
Flag any claim NOT supported by the sources, or that contradicts them.
Respond with exactly "PASS" if fully grounded.
Otherwise respond with "REVISE: <specific issue to fix>"."""


def critique_explanation(explanation: str, sources: list[dict]) -> str:
    source_text = "\n\n".join(s["chunk_text"] for s in sources)
    response = llm.invoke(
        [
            {"role": "system", "content": CRITIQUE_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Source material:\n{source_text}\n\nExplanation to check:\n{explanation}",
            },
        ]
    )
    return response.content.strip()


def explain_with_self_correction(topic: str, max_retries: int = 3) -> dict:
    result = explain_topic(topic)
    for attempt in range(max_retries):
        verdict = critique_explanation(result["explanation"], result["sources"])
        if verdict.upper().startswith("PASS"):
            return result
        print(f"🔁 Critique flagged an issue (attempt {attempt + 1}): {verdict}")
        response = llm.invoke(
            [
                {"role": "system", "content": EXPLAINER_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": f"Your previous explanation had an issue: {verdict}\nRevise it, still grounded only in the source material.",
                },
            ]
        )
        result["explanation"] = response.content

    final_verdict = critique_explanation(result["explanation"], result["sources"])
    if not final_verdict.upper().startswith("PASS"):
        print(f"⚠️ Still unresolved after {max_retries} revisions: {final_verdict}")
    return result


def learn_about(topic: str):
    result = explain_with_self_correction(topic)
    print(result["explanation"])
    print("\n---")
    reaction = input(
        "Your reaction (e.g. 'got it', 'too basic', 'use an analogy', 'I already know this'): "
    )
    store_correction(topic, reaction)
    return reaction

### Assessment Agent

In [7]:
def _parse_json_response(text: str) -> dict:
    cleaned = text.strip().strip("`")
    if cleaned.lower().startswith("json"):
        cleaned = cleaned[4:]
    try:
        return json.loads(cleaned.strip())
    except json.JSONDecodeError:
        raise ValueError(f"Could not parse JSON from model response: {text[:200]}")


QUIZ_SYSTEM_PROMPT = """You are a quiz generator. Given source material on a topic, create exactly {n} short-answer
quiz questions that test genuine understanding, not recall of a single sentence.
Return ONLY valid JSON: a list of objects with keys "question" and "expected_answer" (a concise model answer,
not verbatim source text)."""


def generate_quiz(topic: str, n: int = 3) -> list[dict]:
    sources = retrieve_context(topic, top_k=5)
    source_text = "\n\n".join(s["chunk_text"] for s in sources)
    response = llm.invoke(
        [
            {"role": "system", "content": QUIZ_SYSTEM_PROMPT.format(n=n)},
            {
                "role": "user",
                "content": f"Topic: {topic}\n\nSource material:\n{source_text}",
            },
        ]
    )
    return _parse_json_response(response.content)


GRADE_SYSTEM_PROMPT = """You are grading a learner's answer against a model answer.
Respond with ONLY valid JSON: {"correct": true or false, "feedback": "<one specific sentence>"}"""


def grade_answer(question: str, expected_answer: str, learner_answer: str) -> dict:
    try:
        response = llm.invoke(
            [
                {"role": "system", "content": GRADE_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": f"Question: {question}\nModel answer: {expected_answer}\nLearner's answer: {learner_answer}",
                },
            ]
        )
        return _parse_json_response(response.content)
    except ValueError:
        return {
            "correct": False,
            "feedback": "Grading response couldn't be parsed — treating as unanswered.",
        }


def run_quiz(topic: str, n: int = 3) -> list[dict]:
    questions = generate_quiz(topic, n=n)
    results = []
    for i, q in enumerate(questions, 1):
        print(f"\nQ{i}: {q['question']}")
        answer = input("Your answer: ")
        grade = grade_answer(q["question"], q["expected_answer"], answer)
        print("✅ Correct!" if grade["correct"] else f"❌ {grade['feedback']}")
        results.append({**q, "learner_answer": answer, **grade})
    return results


def store_assessment(topic: str, results: list[dict]):
    correct = sum(r["correct"] for r in results)
    total = len(results)
    missed = [r["question"] for r in results if not r["correct"]]

    summary = f"Scored {correct}/{total} on a quiz about '{topic}'."
    summary += (
        f" Struggled with: {'; '.join(missed)}."
        if missed
        else " Answered all questions correctly."
    )

    mem0_client.add(
        [{"role": "user", "content": summary}],
        user_id=USER_ID,
        metadata={"topic": topic, "type": "assessment", "score": f"{correct}/{total}"},
    )


def assess_topic(topic: str, n: int = 3):
    results = run_quiz(topic, n=n)
    store_assessment(topic, results)
    correct = sum(r["correct"] for r in results)
    print(f"\nFinal score: {correct}/{len(results)}")
    return results

### Research Agent

In [8]:
def make_structured_research_tools():
    collected: list[Paper] = []

    @tool
    def arxiv_search(query: str, max_results: int = 5) -> str:
        """Search arXiv for preprints. Best for recent, cutting-edge ML/CS research."""
        papers = search_arxiv(query, max_results)
        collected.extend(papers)
        return (
            "\n\n".join(
                f"[{p.title}] ({p.year}): {(p.abstract or '')[:300]}" for p in papers
            )
            or "No results found."
        )

    @tool
    def openalex_search(query: str, max_results: int = 5) -> str:
        """Search OpenAlex for academic papers. Broad coverage across all disciplines."""
        papers = search_openalex(query, max_results)
        collected.extend(papers)
        return (
            "\n\n".join(
                f"[{p.title}] ({p.year}): {(p.abstract or '')[:300]}" for p in papers
            )
            or "No results found."
        )

    @tool
    def semantic_scholar_search(query: str, max_results: int = 5) -> str:
        """Search Semantic Scholar for academic papers. Strong for CS/AI topics and citation counts."""
        papers = search_semantic_scholar(query, max_results)
        collected.extend(papers)
        return (
            "\n\n".join(
                f"[{p.title}] ({p.year}): {(p.abstract or '')[:300]}" for p in papers
            )
            or "No results found."
        )

    @tool
    def wikipedia_search(query: str, max_results: int = 2) -> str:
        """Search Wikipedia for background/definitional context on a topic."""
        papers = search_wikipedia(query, max_results)
        collected.extend(papers)
        return (
            "\n\n".join(f"[{p.title}]: {(p.abstract or '')[:300]}" for p in papers)
            or "No results found."
        )

    return collected, [
        arxiv_search,
        openalex_search,
        semantic_scholar_search,
        wikipedia_search,
    ]


tavily_tool = TavilySearch(max_results=5, topic="general")

RESEARCH_AGENT_PROMPT = """You are a research agent. Use the available tools to gather information on the user's topic.
- Use arxiv_search, openalex_search, semantic_scholar_search, and wikipedia_search for structured, storable sources —
  call at least one of these whenever the topic is a research/technical concept, since their results get saved for future use.
- Use tavily_tool for current, practical, or non-academic information not found in academic literature.
Call multiple tools if it helps build a complete picture, then summarize what you found."""


def build_research_agent():
    collected_papers, structured_tools = make_structured_research_tools()
    all_tools = structured_tools + [tavily_tool]
    agent = create_agent(llm, tools=all_tools, system_prompt=RESEARCH_AGENT_PROMPT)
    return agent, collected_papers

### Supervisor

In [9]:
from groq import BadRequestError


class LearnLoopState(TypedDict):
    user_input: str
    intent: Literal["research", "explain", "assess"] | None
    topic: str | None
    papers: list[Paper] | None
    agent_summary: str | None
    has_sources: bool | None
    response: str | None


INTENT_SYSTEM_PROMPT = """Classify the user's request into exactly one intent and extract the topic.
Intents:
- "research": user wants to find/search papers on a topic
- "explain": user wants to learn/understand a concept
- "assess": user wants to be quizzed/tested on a topic

Respond with ONLY valid JSON: {"intent": "research" | "explain" | "assess", "topic": "<the topic, cleaned up>"}"""


def _invoke_research_agent_with_retry(agent, topic: str, max_retries: int = 2):
    for attempt in range(max_retries + 1):
        try:
            return agent.invoke(
                {
                    "messages": [
                        {"role": "user", "content": f"Research this topic: {topic}"}
                    ]
                }
            )
        except BadRequestError as e:
            if "tool_use_failed" in str(e) or "was not in request.tools" in str(e):
                print(
                    f"⚠️ Tool call generation glitch (attempt {attempt + 1}/{max_retries + 1}); retrying..."
                )
                continue
            raise
    return None


def classify_intent(state: LearnLoopState) -> LearnLoopState:
    response = llm.invoke(
        [
            {"role": "system", "content": INTENT_SYSTEM_PROMPT},
            {"role": "user", "content": state["user_input"]},
        ]
    )
    parsed = _parse_json_response(response.content)
    return {**state, "intent": parsed["intent"], "topic": parsed["topic"]}


def check_sources_node(state: LearnLoopState) -> LearnLoopState:
    sources = retrieve_context(state["topic"], top_k=1)
    return {**state, "has_sources": bool(sources)}


def research_agent_node(state: LearnLoopState) -> LearnLoopState:
    agent, collected_papers = build_research_agent()
    result = _invoke_research_agent_with_retry(agent, state["topic"])

    if result is not None:
        return {
            **state,
            "papers": collected_papers,
            "agent_summary": result["messages"][-1].content,
        }

    print(
        "⚠️ Research agent tool-calling was unreliable — falling back to direct search across all sources."
    )
    papers = (
        search_arxiv(state["topic"])
        + search_openalex(state["topic"])
        + search_semantic_scholar(state["topic"])
        + search_wikipedia(state["topic"])
    )
    summary = "\n".join(f"- {p.title} ({p.year}, {p.source})" for p in papers)
    return {
        **state,
        "papers": papers,
        "agent_summary": f"Found {len(papers)} papers (fallback mode):\n{summary}",
    }


def ingest_node(state: LearnLoopState) -> LearnLoopState:
    papers = state.get("papers") or []
    if not papers:
        print(
            "⚠️ No structured papers collected to ingest — agent may have relied only on Wikipedia/Tavily context."
        )
        return state
    inserted = ingest_papers(papers)
    print(f"📥 Ingested {inserted} chunks for '{state['topic']}'")
    return state


def format_research_response(state: LearnLoopState) -> LearnLoopState:
    response = state.get("agent_summary", "")
    if state.get("papers"):
        paper_list = "\n".join(
            f"- {p.title} ({p.year}, {p.source})" for p in state["papers"]
        )
        response += f"\n\nStored for future retrieval:\n{paper_list}"
    return {**state, "response": response}


def explain_node(state: LearnLoopState) -> LearnLoopState:
    result = explain_with_self_correction(state["topic"])
    print(result["explanation"])
    reaction = input("Your reaction: ")
    store_correction(state["topic"], reaction)
    return {**state, "response": result["explanation"]}


def assess_node(state: LearnLoopState) -> LearnLoopState:
    results = run_quiz(state["topic"], n=3)
    store_assessment(state["topic"], results)
    correct = sum(r["correct"] for r in results)
    return {**state, "response": f"Quiz complete: {correct}/{len(results)} correct."}


def route_by_intent(state: LearnLoopState) -> str:
    return "research_agent" if state["intent"] == "research" else "check_sources"


def route_after_check(state: LearnLoopState) -> str:
    return state["intent"] if state["has_sources"] else "research_agent"


def route_after_ingest(state: LearnLoopState) -> str:
    return "format_research" if state["intent"] == "research" else state["intent"]


graph = StateGraph(LearnLoopState)
graph.add_node("classify", classify_intent)
graph.add_node("check_sources", check_sources_node)
graph.add_node("research_agent", research_agent_node)
graph.add_node("ingest", ingest_node)
graph.add_node("format_research", format_research_response)
graph.add_node("explain", explain_node)
graph.add_node("assess", assess_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges(
    "classify",
    route_by_intent,
    {
        "research_agent": "research_agent",
        "check_sources": "check_sources",
    },
)
graph.add_conditional_edges(
    "check_sources",
    route_after_check,
    {
        "explain": "explain",
        "assess": "assess",
        "research_agent": "research_agent",
    },
)
graph.add_edge("research_agent", "ingest")
graph.add_conditional_edges(
    "ingest",
    route_after_ingest,
    {
        "format_research": "format_research",
        "explain": "explain",
        "assess": "assess",
    },
)
graph.add_edge("format_research", END)
graph.add_edge("explain", END)
graph.add_edge("assess", END)

supervisor = graph.compile()

In [10]:
r1 = supervisor.invoke({"user_input": "Research diffusion models"})
print(r1["response"])

📥 Ingested 66 chunks for 'diffusion models'
**Diffusion Models – A Rapidly‑Growing Family of Generative AI Techniques**

---

## 1. What a “diffusion model” is  

In machine‑learning (ML) literature, *diffusion models* (also called **diffusion‑based generative models** or **score‑based generative models**) are latent‑variable generative models that learn to reverse a gradual noising process.  

| Component | Description |
|-----------|-------------|
| **Forward diffusion** | Starting from a real data sample (e.g., an image), Gaussian (or other) noise is added in many small steps until the data is essentially pure noise. This defines a tractable *forward* Markov chain. |
| **Reverse (denoising) process** | A neural network is trained to predict – either the original data or the noise – at each step, thereby learning a *reverse* Markov chain that can turn random noise back into a realistic sample. |
| **Training objective** | Usually a variational bound or a *score‑matching* loss that en

In [11]:
r2 = supervisor.invoke({"user_input": "Explain diffusion models"})
print(r2["response"])

**Diffusion models – a concise, concrete overview**

A diffusion model is a *latent‑variable generative model* that treats data generation as a **diffusion process**.  
In this view, a new datum (e.g., an image, a sound clip, or any high‑dimensional object) is imagined to perform a **random walk with drift** through the space of all possible data points. The model learns how this walk should be orchestrated so that, when we run it in reverse, we end up with realistic samples that follow the same distribution as the original dataset.

---

### Two complementary phases

1. **Forward diffusion (noising) process**  
   - Starting from a real data point, we *gradually add Gaussian noise* in many small steps.  
   - Mathematically this is often described as a **conditional Gaussian noising process**. After enough steps the data become indistinguishable from pure noise.  
   - The forward process is *fixed*; its schedule (how much variance is added at each step) controls the dynamics of the d